# H4: Foraging Profiles and Optimality

Tests whether ω and κ define meaningful profiles, predict survival, and explain overcaution.

- **H4a:** Four ω-κ profiles with distinct earnings
- **H4b:** ω predicts escape on attack trials
- **H4c:** Overcaution is the dominant error; ω predicts who is overcautious

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, os
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import pearsonr, zscore, f_oneway
from pathlib import Path

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.spines.right': False, 'axes.spines.top': False})

REPO_ROOT = Path(os.getcwd())
for _ in range(5):
    if (REPO_ROOT / '.git').exists(): break
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

EXCLUDE = [154, 197, 208]
DATA_DIR = Path('data/exploratory_350/processed/stage5_filtered_data_20260320_191950')

# Load CM2 params
master = pd.read_csv('results/stats/joint_optimal/master_subject_df.csv', index_col='subj')
subjects = master.index.tolist()
master['omega_z'] = zscore(np.log(master['omega']))
master['kappa_z'] = zscore(np.log(master['kappa']))

# Load behavioral data for optimality analysis
beh = pd.read_csv(DATA_DIR / 'behavior_rich.csv', low_memory=False)
beh = beh[~beh['subj'].isin(EXCLUDE)]
cdf = beh[beh['type'] == 1].copy()
cdf['T_round'] = cdf['threat'].round(1)

# Escape rate (survived given attack)
attack = beh[beh['isAttackTrial'] == 1]
escape_rate = attack.groupby('subj').apply(lambda x: (x['trialEndState'] == 'escaped').mean())
master['escape_rate'] = escape_rate

# Earnings
master['earnings'] = beh.groupby('subj')['trialReward'].sum()

# Choice
master['p_heavy'] = cdf.groupby('subj')['choice'].mean()

# Vigor
cells = pd.read_csv('results/stats/vigor_analysis/cell_means.csv')
master['mean_vigor'] = cells.groupby('subj')['mean_rate'].mean()

# Optimality
opt_map = {}
for (T, D), grp in cdf.groupby(['T_round', 'distance_H']):
    er_h = grp[grp['choice']==1]['trialReward'].mean() if (grp['choice']==1).sum()>0 else -99
    er_l = grp[grp['choice']==0]['trialReward'].mean() if (grp['choice']==0).sum()>0 else -99
    opt_map[(T,D)] = 1 if er_h > er_l else 0
cdf['optimal'] = cdf.apply(lambda r: opt_map.get((r['T_round'],r['distance_H']),np.nan), axis=1)
cdf['is_opt'] = (cdf['choice'] == cdf['optimal']).astype(int)
cdf['err_type'] = np.where(cdf['is_opt']==1, 'correct',
                           np.where(cdf['choice']==0, 'overcautious', 'reckless'))
subj_opt = cdf.groupby('subj').agg(
    pct_opt=('is_opt','mean'),
    n_oc=('err_type', lambda x: (x=='overcautious').sum()),
    n_rk=('err_type', lambda x: (x=='reckless').sum()),
    n_err=('is_opt', lambda x: (x==0).sum()))
subj_opt['oc_ratio'] = subj_opt['n_oc'] / subj_opt['n_err'].clip(1)
# Drop any existing optimality columns before join
for c in subj_opt.columns:
    if c in master.columns:
        master = master.drop(columns=[c])
master = master.join(subj_opt, how='left')

print(f'Master: {len(master)} subjects')
print(f'  Escape: {master["escape_rate"].mean():.3f}, Earnings: {master["earnings"].mean():.1f}')

Master: 290 subjects
  Escape: 0.384, Earnings: 6.5


## H4a: Four ω-κ profiles with distinct earnings

In [2]:
om_med = master['omega_z'].median()
kap_med = master['kappa_z'].median()

conditions = [
    (master['omega_z'] < om_med) & (master['kappa_z'] < kap_med),
    (master['omega_z'] >= om_med) & (master['kappa_z'] < kap_med),
    (master['omega_z'] < om_med) & (master['kappa_z'] >= kap_med),
    (master['omega_z'] >= om_med) & (master['kappa_z'] >= kap_med),
]
labels = ['Resilient', 'Strategic', 'Reckless', 'Helpless']
master['profile'] = np.select(conditions, labels, default='?')

print('H4a — Behavioral Profiles')
print('=' * 70)
print(f'{"Profile":<12} {"N":>4} {"P(H)":>7} {"Escape":>8} {"Vigor":>8} {"Earn":>8}')
print('-' * 50)
for lab in ['Resilient', 'Strategic', 'Reckless', 'Helpless']:
    s = master[master['profile'] == lab]
    print(f'{lab:<12} {len(s):>4} {s["p_heavy"].mean():>7.3f} {s["escape_rate"].mean():>8.3f} '
          f'{s["mean_vigor"].mean():>8.3f} {s["earnings"].mean():>+8.1f}')

groups = [master.loc[master['profile']==q, 'earnings'].dropna().values for q in labels]
F, p = f_oneway(*groups)
passed = p < 0.01
print(f'\n  ANOVA on earnings: F={F:.1f}, p={p:.4f}')
print(f'  H4a: {"PASS ✓" if passed else "FAIL ✗"} (threshold: p < .01)')

H4a — Behavioral Profiles
Profile         N    P(H)   Escape    Vigor     Earn
--------------------------------------------------
Resilient     103   0.608    0.337    1.026    +16.1
Strategic      42   0.352    0.450    1.253    +22.5
Reckless       42   0.529    0.317    0.796    -10.8
Helpless      103   0.242    0.431    0.867     -2.6

  ANOVA on earnings: F=1.8, p=0.1435
  H4a: FAIL ✗ (threshold: p < .01)


## H4b: ω predicts escape on attack trials

In [3]:
X = sm.add_constant(master[['omega_z', 'kappa_z']].values)
y = master['escape_rate'].values
ols = sm.OLS(y, X).fit()

b_om = ols.params[1]; p_om = ols.pvalues[1]
b_kap = ols.params[2]; p_kap = ols.pvalues[2]
passed = b_om > 0 and p_om < 0.01

print('H4b — ω predicts escape')
print('=' * 50)
print(f'  ω: β={b_om:+.4f}, p={p_om:.4f}')
print(f'  κ: β={b_kap:+.4f}, p={p_kap:.4f}')
print(f'  R²={ols.rsquared:.4f}')
print(f'  H4b: {"PASS ✓" if passed else "FAIL ✗"} (threshold: ω β > 0, p < .01)')

H4b — ω predicts escape
  ω: β=+0.0603, p=0.0002
  κ: β=-0.0037, p=0.8150
  R²=0.0572
  H4b: PASS ✓ (threshold: ω β > 0, p < .01)


## H4c: Overcaution is the dominant error

In [4]:
oc_pct = master['oc_ratio'].mean() * 100
r_oc, p_oc = pearsonr(master['omega_z'], master['oc_ratio'])

pass_pct = oc_pct > 65
pass_r = r_oc > 0.30 and p_oc < 0.01
passed = pass_pct and pass_r

print('H4c — Overcaution')
print('=' * 50)
print(f'  Overcautious errors: {oc_pct:.0f}% of all errors {"✓" if pass_pct else "✗"} (threshold: >65%)')
print(f'  r(ω, overcaution ratio) = {r_oc:+.3f} (p={p_oc:.4f}) {"✓" if pass_r else "✗"} (threshold: r>.30, p<.01)')
print(f'  H4c: {"PASS ✓" if passed else "FAIL ✗"}')

H4c — Overcaution
  Overcautious errors: 79% of all errors ✓ (threshold: >65%)
  r(ω, overcaution ratio) = +0.810 (p=0.0000) ✓ (threshold: r>.30, p<.01)
  H4c: PASS ✓


## Summary

In [5]:
# Collect results
groups = [master.loc[master['profile']==q, 'earnings'].dropna().values for q in labels]
F_val, p_val = f_oneway(*groups)

tests = [
    ('H4a', 'Profile ANOVA (earnings)', 'p < .01',
     f'F={F_val:.1f}, p={p_val:.4f}', p_val < 0.01),
    ('H4b', 'ω → escape', 'β > 0, p < .01',
     f'β={b_om:+.4f}, p={p_om:.4f}', b_om > 0 and p_om < 0.01),
    ('H4c', 'Overcaution %', '> 65%',
     f'{oc_pct:.0f}%', oc_pct > 65),
    ('H4c', 'ω → overcaution', 'r > .30, p < .01',
     f'r={r_oc:+.3f}, p={p_oc:.4f}', r_oc > 0.30 and p_oc < 0.01),
]

print('H4 SUMMARY')
print('=' * 70)
print(f'{"Hyp":<6} {"Test":<28} {"Threshold":<18} {"Result":<25} {"Pass":<5}')
print('-' * 80)
for hyp, test, thresh, result, passed in tests:
    print(f'{hyp:<6} {test:<28} {thresh:<18} {result:<25} {"✓" if passed else "✗"}')

n_pass = sum(1 for *_, p in tests if p)
print(f'\n{n_pass}/{len(tests)} tests pass.')

H4 SUMMARY
Hyp    Test                         Threshold          Result                    Pass 
--------------------------------------------------------------------------------
H4a    Profile ANOVA (earnings)     p < .01            F=1.8, p=0.1435           ✗
H4b    ω → escape                   β > 0, p < .01     β=+0.0603, p=0.0002       ✓
H4c    Overcaution %                > 65%              79%                       ✓
H4c    ω → overcaution              r > .30, p < .01   r=+0.810, p=0.0000        ✓

3/4 tests pass.
